### Embedding Comparison


In [3]:
from pathlib import Path
from gensim.models import Word2Vec, FastText

from utils import load_sets, clean, tokenise

In [4]:
train_set, val_set, test_set = load_sets()

print("Train:", len(train_set))
print("Val:", len(val_set))
print("Test:", len(test_set))

train_set.head()

Train: 199222
Val: 22136
Test: 24596


,input,response
0,"hello, my little friends. where can i find a c...",nfi #windows
1,i heard fedora was better than ubuntu,than join #fedora you hear things join #ubunt...
2,"if your doing hardware hacking yes, if your a ...",thanks. :)
3,hi everyone! that joke isn't funny anymore.,why? cuz he died?
4,so where is the boot loader like grub stored i...,<PATH> is inside of / if there is no separate...


In [5]:
def build_sentences(df):
    sentences = []

    for _, row in df.iterrows():
        input_text = clean(str(row["input"]))
        response_text = clean(str(row["response"]))

        input_tokens = tokenise(input_text)
        response_tokens = tokenise(response_text)

        if input_tokens:
            sentences.append(input_tokens)
        if response_tokens:
            sentences.append(response_tokens)

    return sentences

sentences = build_sentences(train_set)

print("Number of tokenized sentences:", len(sentences))
print("Example:", sentences[0][:20] if sentences else "No data")

Number of tokenized sentences: 398444
Example: ['hello,', 'my', 'little', 'friends.', 'where', 'can', 'i', 'find', 'a', 'channel', 'for', 'internetexplorer.application', '?', '(activex)']


In [6]:


Path("embeddings").mkdir(exist_ok=True)

w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1
)

w2v_model.save("embeddings/word2vec.model")

print("Word2Vec trained and saved as word2vec.model")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Word2Vec trained and saved as word2vec.model


In [7]:


Path("embeddings").mkdir(exist_ok=True)

fasttext_model = FastText(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1
)

fasttext_model.save("embeddings/fasttext.model")

print("FastText trained and saved as fasttext.model")

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


FastText trained and saved as fasttext.model


### Replicating Model A loading

In [8]:
from utils import load_sets, load_vocab, load_config, get_dataloaders

train_set, val_set, test_set = load_sets()
vocab = load_vocab()
config = load_config()

train_loader, val_loader, test_loader = get_dataloaders(
    train_set, val_set, test_set,
    vocab,
    config['MAX_LENGTH'],
    config['BATCH_SIZE']
)

### Pretrained from same vocab

In [9]:
import numpy as np
import torch
from gensim.models import Word2Vec, FastText

EMBEDDING_TYPE = "scratch"   # "scratch", "word2vec", "fasttext"

w2v_model = Word2Vec.load("embeddings/word2vec.model")
ft_model = FastText.load("embeddings/fasttext.model")

def build_embedding_matrix(vocab, model, embed_dim, pad_idx):
    matrix = np.random.normal(scale=0.6, size=(len(vocab), embed_dim)).astype(np.float32)
    matrix[pad_idx] = np.zeros(embed_dim, dtype=np.float32)

    for token, idx in vocab.items():
        if token in model.wv:
            matrix[idx] = model.wv[token]

    return torch.tensor(matrix, dtype=torch.float32)

ENCODER_EMBED_DIM = 100
DECODER_EMBED_DIM = 100

w2v_matrix_enc = build_embedding_matrix(vocab, w2v_model, ENCODER_EMBED_DIM, config["PAD_IDX"])
ft_matrix_enc  = build_embedding_matrix(vocab, ft_model,  ENCODER_EMBED_DIM, config["PAD_IDX"])

w2v_matrix_dec = build_embedding_matrix(vocab, w2v_model, DECODER_EMBED_DIM, config["PAD_IDX"])
ft_matrix_dec  = build_embedding_matrix(vocab, ft_model,  DECODER_EMBED_DIM, config["PAD_IDX"])

### Encoder/Decoder/Seq 2 Seq

In [10]:
from encoder import Encoder
from decoder import Decoder
from seq2Seq import Seq2Seq



### Train Eval

In [12]:
from tqdm import tqdm
import torch
import torch.nn as nn

def train_epoch(model, loader, optimizer, criterion, device, clip=1.0, teacher_forcing_proba=0.5):
    model.train()
    total_loss = 0

    progress = tqdm(loader, desc="Training", leave=False)

    for src, tgt_in, tgt_out in progress:
        src = src.to(device)
        tgt_in = tgt_in.to(device)
        tgt_out = tgt_out.to(device)

        optimizer.zero_grad()

        output = model(src, tgt_in, teacher_forcing_proba=teacher_forcing_proba)
        output = output[:, 1:, :].reshape(-1, output.shape[-1])
        tgt_out = tgt_out[:, 1:].reshape(-1)

        loss = criterion(output, tgt_out)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        total_loss += loss.item()
        progress.set_postfix(loss=loss.item())

    return total_loss / len(loader)


def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0

    progress = tqdm(loader, desc="Validation", leave=False)

    with torch.no_grad():
        for src, tgt_in, tgt_out in progress:
            src = src.to(device)
            tgt_in = tgt_in.to(device)
            tgt_out = tgt_out.to(device)

            output = model(src, tgt_in, teacher_forcing_proba=0.0)
            output = output[:, 1:, :].reshape(-1, output.shape[-1])
            tgt_out = tgt_out[:, 1:].reshape(-1)

            loss = criterion(output, tgt_out)
            total_loss += loss.item()
            progress.set_postfix(loss=loss.item())

    return total_loss / len(loader)

### Model Building

In [13]:
def make_model(embedding_type, vocab, config, device):
    ENCODER_HIDDEN_DIM = 256
    ENCODER_NUM_LAYERS = 2
    ENCODER_DROPOUT_PROBA = 0.3

    DECODER_HIDDEN_DIM = 256
    DECODER_NUM_LAYERS = 2
    DECODER_DROPOUT_PROBA = 0.3

    model = Seq2Seq(
        encoder=Encoder(
            vocab_size=config["VOCAB_SIZE"],
            embed_dim=ENCODER_EMBED_DIM,
            padding_idx=config["PAD_IDX"],
            hidden_dim=ENCODER_HIDDEN_DIM,
            num_layers=ENCODER_NUM_LAYERS,
            dropout_proba=ENCODER_DROPOUT_PROBA,
            use_attention=False
        ),
        decoder=Decoder(
            vocab_size=config["VOCAB_SIZE"],
            embed_dim=DECODER_EMBED_DIM,
            padding_idx=config["PAD_IDX"],
            hidden_dim=DECODER_HIDDEN_DIM,
            num_layers=DECODER_NUM_LAYERS,
            dropout_proba=DECODER_DROPOUT_PROBA,
            use_attention=False
        ),
        device=device,
        use_attention=False
    ).to(device)

    with torch.no_grad():
        if embedding_type == "word2vec":
            model.encoder.embedding.weight.copy_(w2v_matrix_enc)
            model.decoder.embedding.weight.copy_(w2v_matrix_dec)

        elif embedding_type == "fasttext":
            model.encoder.embedding.weight.copy_(ft_matrix_enc)
            model.decoder.embedding.weight.copy_(ft_matrix_dec)

        elif embedding_type == "scratch":
            pass

        else:
            raise ValueError(f"Unknown embedding_type: {embedding_type}")

    return model

### Experiment Runner

In [18]:
EPOCHS = 10
CLIP_MAX_NORM = 1.0
TEACHER_FORCING_PROBA = 0.5

In [ ]:
embedding_types = ["scratch", "word2vec", "fasttext"]
results = []

for emb_type in embedding_types:
    print(f"\nTraining with {emb_type} embeddings")

    model = make_model(emb_type, vocab, config, device)

    criterion = nn.CrossEntropyLoss(ignore_index=config["PAD_IDX"])
    optimiser = Adam(model.parameters(), lr=1e-3)

    epoch_train_losses = []
    epoch_val_losses = []

    for epoch in range(EPOCHS):
        train_loss = train_epoch(
            model,
            train_loader,
            optimiser,
            criterion,
            device,
            CLIP_MAX_NORM,
            TEACHER_FORCING_PROBA
        )

        val_loss = evaluate_epoch(
            model,
            val_loader,
            criterion,
            device
        )

        epoch_train_losses.append(train_loss)
        epoch_val_losses.append(val_loss)

        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    results.append({
        "Embedding": emb_type,
        "Final Train Loss": epoch_train_losses[-1],
        "Final Val Loss": epoch_val_losses[-1],
        "Best Val Loss": min(epoch_val_losses)
    })


Training with scratch embeddings


Epoch 1/10 | Train Loss: 6.2562 | Val Loss: 6.1865


Epoch 2/10 | Train Loss: 6.0268 | Val Loss: 6.1256


Training:  12%|██▏                | 365/3113 [01:56<15:49,  2.89it/s, loss=5.73]

### Comparison

In [ ]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df.sort_values("Best Val Loss")